# 05 — Event Study: Weather Shocks & Price Response

**Objective**: Measure the cumulative abnormal return (CAR) of coffee futures around heatwave, drought, and compound weather events across Brazil's mesoregions.

**Method**: For each shock event, compare actual returns to a pre-event trend from −30 to +90 days.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import DATA_PROCESSED_DIR
from src.data.merge_data import load_combined
from src.analysis.detect_shocks import detect_all, load_events
from src.analysis.event_study import run_event_study, load_events as load_shock_events
from src.visualization.correlation_plots import (
    car_by_event_type,
    car_by_production_quartile,
)

combined = load_combined()
events = load_shock_events()
car = pd.read_parquet(DATA_PROCESSED_DIR / "event_study_results.parquet")

print(f"Combined data: {len(combined)} days")
print(f"Shock events: {len(events)}")
print(f"CAR records: {len(car)}")

## 5.1 Shock Events Overview

Breakdown by event type and top mesoregions.

In [ ]:
print("=== Events by Type ===")
print(events["event_type"].value_counts().to_string())
print()
print("=== Events by Mesoregion (Top 10) ===")
print(events.groupby("meso_name").size().sort_values(ascending=False).head(10).to_string())
print()
print("=== Avg Duration by Event Type ===")
print(events.groupby("event_type")["duration_days"].mean().round(1).to_string())

## 5.2 Cumulative Abnormal Return by Event Type

How does the market respond to heatwaves vs. droughts vs. compound events over 10/30/60/90 days?

In [ ]:
car_by_event_type().show()

## 5.3 Statistical Significance

For each event type × horizon combination, test if mean CAR differs significantly from zero.

In [ ]:
print("=== T-Test: CAR vs Zero ===\n")
for etype in ["heatwave", "drought", "compound"]:
    subset = car[car["event_type"] == etype]
    if subset.empty:
        continue
    print(f"  {etype.upper()}:")
    for horizon in ["10d", "30d", "60d", "90d"]:
        h = subset[subset["horizon"] == horizon]["car_pct"]
        if h.empty:
            continue
        mean_car = h.mean()
        t_stat, p_val = stats.ttest_1samp(h, 0)
        sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
        print(f"    {horizon:>4}: mean CAR = {mean_car:+.2f}%  (t={t_stat:.2f}, p={p_val:.4f}){sig}")

## 5.4 Production-Weighted Impact

Does production tonnage matter? Group events by production quartile and compare 90-day CAR.

In [ ]:
car_by_production_quartile().show()

## 5.5 Top Mesoregions by Impact

Which mesoregions' weather shocks produce the strongest price response at 90 days?

In [ ]:
q90 = car[car["horizon"] == "90d"].copy()
meso_impact = q90.groupby(["meso_name", "state_name"]).agg(
    mean_car=("car_pct", "mean"),
    event_count=("car_pct", "count"),
    production=("production_tonnes", "max"),
).sort_values("mean_car", ascending=False).reset_index()

meso_impact["prod_rank"] = meso_impact["production"].rank(ascending=False).astype(int)
meso_impact.head(10)

## 5.6 Event Study: Full Pipeline

Re-run the event study from scratch.

In [ ]:
run_event_study()

## Key Observations

- Fill in your observations here after running the cells above.